# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [3]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

In [8]:
placement_agent = Agent(
    name="Placement Trainer",
    instructions="""
    You are a helpful guide for students preparing for job placements.
    Help with resumes, aptitude tests, coding and technical interviews,
    behavioral questions, and job-search planning.
    Give concise, practical answers with clear next steps. Tailor advice
    to the student's experience and target role. Ask a brief clarifying
    question when important context is missing. Be honest and never
    guarantee placement outcomes.
    """,
)

Let's execute the Agent:

In [4]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are healthy as part of a balanced diet.
    
    Key points:
    - Nutrients: good source of potassium, vitamin B6, vitamin C, and fiber.
    - Health benefits: supports heart health, digestion, and energy; resistant starch in unripe bananas may aid gut health.
    - Calories: about 90–110 kcal per medium fruit; low fat.
    - Considerations: relatively high natural sugars; portion matters if you’re watching sugar/carb intake or diabetes.
    
    Tips:
    - One medium banana daily fits many diets.
    - Riper bananas have sweeter taste and softer texture; unripe have more resistant starch.
    - Pair with protein or fats (e.g., peanut butter) to improve satiety.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [5]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Bananas are nutritious and generally healthy as part of a balanced diet.

Key points:
- Great source of potassium (about 420 mg per medium banana), which supports blood pressure and heart function.
- Good fiber (roughly 3 g) and about 14 g natural sugars; provides steady energy.
- Vitamins/minerals: vitamin B6, vitamin C, and some magnesium.
- About 105 kcal per medium banana.

Tips:
- Pair with protein (e.g., yogurt, peanut butter) for balanced meals.
- Choose ripe for easier digestion; greener for slower sugar release.
- Safe for most people; avoid only if you have a banana allergy or specific medical constraints.

Overall: Healthy, convenient fruit to include regularly in a varied diet.

In [12]:
placement_agent = Agent(
    name="Placement Trainer",
    instructions="""
    You are a helpful guide for students preparing for job placements.
    Help with resumes, aptitude tests, coding and technical interviews,
    behavioral questions, and job-search planning.
    Give concise, practical answers with clear next steps. Tailor advice
    to the student's experience and target role. Ask a brief clarifying
    question when important context is missing. Be honest and never
    guarantee placement outcomes.
    """,
)



In [14]:
with trace("Simple placement Agent"):
    result = await Runner.run(placement_agent, "How should i prepare for interviews in 30 days?")

print(result)

RunResult:
- Last agent: Agent(name="Placement Trainer", ...)
- Final output (str):
    Great goal. Here’s a practical 30-day plan you can tailor to your target role (tech/engineering is assumed). It’s a tight schedule, so aim for 1–2 focused blocks per weekday and lighter practice on weekends.
    
    Key setup (days 1–2)
    - Clarify target roles and companies, note 3–5 must-know skills.
    - Prepare a concise 1-page resume and a short elevator pitch (24–30 seconds).
    - Create a blank STAR stories sheet: Situation, Task, Action, Result (quantify outcomes).
    
    Week 1 — Foundations and quick wins (days 1–7)
    - Day 1: Resume polish + LinkedIn/GitHub alignment. Add 2–3 project highlights with impact metrics.
    - Day 2: Tech fundamentals refresher. Pick your stack: data structures, big-O, common algorithms.
    - Day 3: Behavioral prep start. Write 5 STAR stories (teams, conflict, ownership, failure, impact).
    - Day 4: Coding practice (30–45 min). Pick 3 medium problem

In [ ]:
student_agent = Agent(
    name="student management",
    instructions="""
    You are a helpful assistant managing the student data.
    You give concise answers.
    """,
)




In [16]:
with trace("Simple student management Agent"):
    result = await Runner.run(student_agent, "How should i manage my student data?")

print(result)

RunResult:
- Last agent: Agent(name="student management", ...)
- Final output (str):
    Here’s a concise guide to manage student data well:
    
    - Inventory
      - List what data you collect (names, IDs, contact, grades, health, attendance, etc.) and why.
      - Note where it’s stored (systems, spreadsheets, paper files).
    
    - Privacy & compliance
      - Know applicable laws (e.g., FERPA in the U.S., GDPR if applicable).
      - Obtain necessary consent/consent management; minimize data collection.
      - Define a data retention schedule; securely destroy data you no longer need.
    
    - Access & security
      - Use role-based access. Limit who can view/edit sensitive data.
      - Enforce strong authentication and encryption at rest/in transit.
      - Keep audit logs of data access and changes.
    
    - Data quality & governance
      - Use unique student IDs; keep data consistent across systems.
      - Regularly clean and de-duplicate data; correct errors promp

_Good Job!_